In [1]:













%pip install google-generativeai

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 2.5 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.3 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---- ---------------

In [10]:
import os
import csv
import time
import getpass

import google.generativeai as genai

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------

ARCHIVO_ENTRADA = "respuestas_encuesta.csv"
ARCHIVO_SALIDA = "resultados_sentimiento.csv"
COLUMNA_RESPUESTA = "respuesta"   
COLUMNA_ID = "id"                 
MODELO = "gemini-2.5-flash-lite"       # modelo económico, apto para free tier (jul 2026)
MODELOS_ALTERNATIVOS = [
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-3-flash-lite",
    "gemini-3-flash-preview",
    "gemini-3.5-flash",
]
PAUSA_ENTRE_LLAMADAS = 4.5        # segundos entre llamadas para respetar límites del free tier

ETIQUETAS_VALIDAS = {"Positivo", "Negativo", "Neutro"}

# ---------------------------------------------------------------------------
# Configurar la API de Gemini
# ---------------------------------------------------------------------------

def obtener_api_key() -> str:
    """Obtiene la API key desde una variable de entorno o la pide de forma interactiva."""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("No se encontró la variable de entorno GEMINI_API_KEY.")
        api_key = getpass.getpass("Ingresa tu API key de Gemini (no se mostrará en pantalla): ").strip()
    if not api_key:
        raise ValueError(
            "No se proporcionó una API key. Consíguela gratis en https://aistudio.google.com/apikey"
        )
    return api_key


def elegir_modelo_disponible(preferido: str, alternativos: list) -> str:
    """
    Prueba una llamada real de generateContent contra cada modelo candidato
    (empezando por el preferido) y se queda con el primero que funcione.
    No basta con revisar ListModels: un modelo puede seguir listado ahí
    pero devolver 404 igualmente para API keys nuevas.
    """
    candidatos = [preferido] + [m for m in alternativos if m != preferido]

    for candidato in candidatos:
        try:
            m = genai.GenerativeModel(candidato)
            m.generate_content("Responde solo con: ok")
            if candidato != preferido:
                print(f"[Aviso] '{preferido}' no está disponible para tu API key. Usando '{candidato}' en su lugar.")
            return candidato
        except Exception as e:
            print(f"[Aviso] Modelo '{candidato}' no disponible ({e}). Probando siguiente opción...")

    raise RuntimeError(
        f"Ninguno de los modelos candidatos {candidatos} funcionó con tu API key.\n"
        "Verifica en https://aistudio.google.com/apikey qué modelos tiene habilitados tu cuenta, "
        "o ejecuta genai.list_models() para ver la lista completa."
    )


API_KEY = obtener_api_key()
genai.configure(api_key=API_KEY)
MODELO = elegir_modelo_disponible(MODELO, MODELOS_ALTERNATIVOS)
print(f"Usando modelo: {MODELO}\n")
modelo = genai.GenerativeModel(MODELO)


def clasificar_sentimiento(texto: str, reintentos: int = 3) -> str:
    """
    Envía el texto a Gemini y devuelve una de: 'Positivo', 'Negativo', 'Neutro'.
    Reintenta ante errores transitorios (rate limit, timeouts, etc.).
    """
    prompt = (
        "Clasifica el sentimiento del siguiente texto en una sola palabra, "
        "elige exactamente entre: Positivo, Negativo o Neutro. "
        "No agregues explicaciones ni puntuación adicional.\n\n"
        f"Texto: \"{texto}\"\n\n"
        "Sentimiento:"
    )

    for intento in range(1, reintentos + 1):
        try:
            respuesta = modelo.generate_content(prompt)
            etiqueta = respuesta.text.strip().splitlines()[0].strip()

            # Normalizar por si el modelo devuelve variaciones de mayúsculas/minúsculas
            etiqueta_normalizada = etiqueta.capitalize()

            if etiqueta_normalizada in ETIQUETAS_VALIDAS:
                return etiqueta_normalizada

           
            for candidata in ETIQUETAS_VALIDAS:
                if candidata.lower() in etiqueta.lower():
                    return candidata

            print(f"  [Aviso] Respuesta inesperada del modelo: '{etiqueta}'. Se marcará como 'Neutro'.")
            return "Neutro"

        except Exception as e:
            print(f"  [Intento {intento}/{reintentos}] Error al llamar a Gemini: {e}")
            if intento < reintentos:
                time.sleep(5 * intento)  
            else:
                print("  No se pudo clasificar esta respuesta. Se marcará como 'Neutro'.")
                return "Neutro"


def guardar_resultados(resultados):
    """Escribe (o sobrescribe) el CSV de salida con lo procesado hasta el momento."""
    with open(ARCHIVO_SALIDA, "w", newline="", encoding="utf-8") as f_out:
        escritor = csv.DictWriter(f_out, fieldnames=["id", "respuesta", "sentimiento"])
        escritor.writeheader()
        escritor.writerows(resultados)


def imprimir_resumen(resultados, conteo):
    print("\n" + "=" * 40)
    print("RESUMEN DE CLASIFICACIÓN")
    print("=" * 40)
    print(f"Total de respuestas procesadas: {len(resultados)}")
    print(f"  Positivas: {conteo['Positivo']}")
    print(f"  Negativas: {conteo['Negativo']}")
    print(f"  Neutras:   {conteo['Neutro']}")
    print(f"\nResultados guardados en: {ARCHIVO_SALIDA}")


def main():
    if not os.path.exists(ARCHIVO_ENTRADA):
        raise FileNotFoundError(
            f"No se encontró el archivo '{ARCHIVO_ENTRADA}'. "
            "Verifica que esté en la misma carpeta que el notebook, "
            "o usa una ruta completa, por ejemplo r'C:\\Users\\Enrique\\Documents\\respuestas_encuesta.csv'."
        )

    with open(ARCHIVO_ENTRADA, newline="", encoding="utf-8") as f_in:
        lector = csv.DictReader(f_in)

        if COLUMNA_RESPUESTA not in (lector.fieldnames or []):
            raise ValueError(
                f"El archivo debe tener una columna llamada '{COLUMNA_RESPUESTA}'. "
                f"Columnas encontradas: {lector.fieldnames}"
            )

        filas = list(lector)

    total = len(filas)
    print(f"Se encontraron {total} respuestas en '{ARCHIVO_ENTRADA}'. Iniciando clasificación...\n")

    resultados = []
    conteo = {"Positivo": 0, "Negativo": 0, "Neutro": 0}

    try:
        for i, fila in enumerate(filas, start=1):
            texto = (fila.get(COLUMNA_RESPUESTA) or "").strip()
            id_respuesta = fila.get(COLUMNA_ID, i)

            if not texto:
                print(f"[{i}/{total}] Respuesta vacía, se omite (se marca como Neutro).")
                sentimiento = "Neutro"
            else:
                print(f"[{i}/{total}] Clasificando respuesta id={id_respuesta}...")
                sentimiento = clasificar_sentimiento(texto)

            conteo[sentimiento] += 1
            resultados.append({
                "id": id_respuesta,
                "respuesta": texto,
                "sentimiento": sentimiento,
            })

            # Guardado incremental: si algo falla o se interrumpe el kernel, no se pierde el progreso ya hecho.
            guardar_resultados(resultados)

            # Pausa para respetar los límites de la API gratuita
            if i < total:
                time.sleep(PAUSA_ENTRE_LLAMADAS)

    except KeyboardInterrupt:
        print("\n[Interrumpido por el usuario] Se guardó el progreso parcial.")

    imprimir_resumen(resultados, conteo)


No se encontró la variable de entorno GEMINI_API_KEY.


Ingresa tu API key de Gemini (no se mostrará en pantalla):  ········


[Aviso] Modelo 'gemini-2.5-flash-lite' no disponible (404 This model models/gemini-2.5-flash-lite is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.). Probando siguiente opción...
[Aviso] Modelo 'gemini-2.5-flash' no disponible (404 This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.). Probando siguiente opción...
[Aviso] Modelo 'gemini-3-flash-lite' no disponible (404 models/gemini-3-flash-lite is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.). Probando siguiente opción...
[Aviso] 'gemini-2.5-flash-lite' no está disponible para tu API key. Usando 'gemini-3-flash-preview' en su lugar.
Usando modelo: gemini-3-flash-preview



In [11]:
main()

Se encontraron 20 respuestas en 'respuestas_encuesta.csv'. Iniciando clasificación...

[1/20] Clasificando respuesta id=1...
[2/20] Clasificando respuesta id=2...
[3/20] Clasificando respuesta id=3...
[4/20] Clasificando respuesta id=4...
[5/20] Clasificando respuesta id=5...
[6/20] Clasificando respuesta id=6...
[7/20] Clasificando respuesta id=7...
[8/20] Clasificando respuesta id=8...
[9/20] Clasificando respuesta id=9...
[10/20] Clasificando respuesta id=10...
[11/20] Clasificando respuesta id=11...
[12/20] Clasificando respuesta id=12...
  [Intento 1/3] Error al llamar a Gemini: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash
Please retry in 16.70348784s. [l

In [ ]:
# ---------------------------------------------------------------------------
# Comentario final
# ---------------------------------------------------------------------------
# Cómo estructuré mi prompt:
# Lo diseñé para que fuera claro y restrictivo: pedí que la salida fuera únicamente una palabra entre "Positivo", "Negativo" o "Neutro", sin
# explicaciones adicionales. Esto ayudó a mantener consistencia y evitar respuestas largas o ambiguas.

# Dificultades encontradas:
# Durante la ejecución tuve problemas con la disponibilidad de modelos:
# varios aparecían listados pero devolvían errores 404, por lo que tuve que probar alternativas hasta llegar a 'gemini-3-flash-preview'.
# También me encontré con límites de cuota en el free tier (error 429), lo que interrumpió algunas llamadas y consumió mis tokens más rápido
# de lo esperado. Para continuar, usé una cuenta alterna y me apoyé en Claude para ajustar la selección.

# Qué mejoraría si tuviera más tiempo:
# Implementaría un sistema de control de tasa más inteligente para evitar exceder los límites de la API gratuita, quizá con colas o batch processing.
# También añadiría un caché local para no repetir llamadas sobre textos ya procesados y optimizar el uso de tokens. Finalmente, exploraría modelos 
# más recientes y estables para asegurar mayor confiabilidad.